# LinReg: Additive linear baseline

Each transformed feature block has its own linear map. It is the transparent baseline for deciding whether nonlinear shape functions are necessary.


## Model


For feature blocks $x_j\in\mathbb{R}^{p_j}$,

$$
\eta(x)=\beta_0+\sum_{j=1}^{d} w_j^\top x_j.
$$

Every $w_j^\top x_j$ is returned as an additive contribution.


## Shared estimator API

All neural estimators use `fit`, `predict`, `score`, `evaluate`, and
`predict_components`. The component result reconstructs predictions on the link
scale and supports shared term-importance and plotting utilities. Constructor
options such as `numerical_method` and `categorical_method` are forwarded to
PreTab and are fitted on training rows only.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(7)
n = 180
X = pd.DataFrame({
    "x1": rng.uniform(-1.0, 1.0, n),
    "x2": rng.normal(size=n),
    "group": rng.choice(["a", "b", "c"], size=n),
})
y = (
    np.sin(np.pi * X["x1"])
    + 0.35 * X["x2"] ** 2
    + 0.30 * (X["group"] == "b")
    + rng.normal(0.0, 0.12, n)
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=7
)

# Set True to run the small fit and all fitted-model demonstrations.
RUN_TRAINING = False


## Construct the estimator


In [ ]:
from nampy.models import LinRegClassifier, LinRegLSS, LinRegRegressor


model = LinRegRegressor(
    intercept=True,
    numerical_method="standardization",
    categorical_method="one-hot",
)
model.get_params(deep=False)


## Fit and inspect

Enable `RUN_TRAINING` above for a short demonstration. Real work should use a
larger validation set, enough epochs, and early stopping.


In [ ]:
if RUN_TRAINING:
    model.fit(
        X_train,
        y_train,
        max_epochs=3,
        batch_size=64,
        random_state=7,
        logger=False,
        enable_progress_bar=False,
        enable_model_summary=False,
    )
    predictions = model.predict(X_test)
    r2 = model.score(X_test, y_test)
    metrics = model.evaluate(X_test, y_test)
    components = model.predict_components(X_test, center=True)
    components.validate_additive_reconstruction()
    display({"R2": r2, **metrics})
    display(model.term_importance(X_test).head())


## Model-specific controls

`intercept` controls the global bias. PreTab settings determine whether a source feature enters as one scalar or an expanded block.


In [ ]:
model.set_params(intercept=False)
model.set_params(intercept=True)
if RUN_TRAINING:
    display(model.predict_components(X_test).terms.keys())


## Task variants and limits

`LinRegRegressor` uses $R^2$ scoring; `LinRegClassifier` exposes `predict_proba`; `LinRegLSS(family=...)` predicts distribution parameters.
